In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_sistema_horario AS

WITH horas AS (
    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_generacion_real

    UNION

    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_disponibilidad_planta

    UNION

    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_demanda_real

    UNION

    SELECT fecha_hora
    FROM observatorio_dev.gold.fact_precio_bolsa
),

generacion AS (
    SELECT
        fecha_hora,
        SUM(generacion_real_kwh) AS generacion_real_kwh
    FROM observatorio_dev.gold.fact_generacion_real
    GROUP BY fecha_hora
),

disponibilidad AS (
    SELECT
        fecha_hora,
        SUM(disponibilidad_real_kwh)
            AS disponibilidad_real_kwh
    FROM observatorio_dev.gold.fact_disponibilidad_planta
    GROUP BY fecha_hora
),

demanda AS (
    SELECT
        fecha_hora,

        SUM(demanda_real_kwh)
            AS demanda_total_kwh,

        SUM(
            CASE
                WHEN tipo_mercado = 'REGULADO'
                THEN demanda_real_kwh
                ELSE 0
            END
        ) AS demanda_regulada_kwh,

        SUM(
            CASE
                WHEN tipo_mercado = 'NO REGULADO'
                THEN demanda_real_kwh
                ELSE 0
            END
        ) AS demanda_no_regulada_kwh

    FROM observatorio_dev.gold.fact_demanda_real
    GROUP BY fecha_hora
),

precio AS (
    SELECT
        fecha_hora,
        precio_bolsa_nacional_cop_kwh,
        precio_bolsa_internacional_cop_kwh,
        precio_bolsa_tie_cop_kwh
    FROM observatorio_dev.gold.fact_precio_bolsa
)

SELECT
    CAST(
        DATE_FORMAT(h.fecha_hora, 'yyyyMMdd')
        AS INT
    ) AS fecha_key,

    CAST(HOUR(h.fecha_hora) + 1 AS TINYINT)
        AS periodo_key,

    h.fecha_hora,
    CAST(h.fecha_hora AS DATE) AS fecha,
    YEAR(h.fecha_hora) AS anio,
    MONTH(h.fecha_hora) AS mes_numero,
    DAYOFWEEK(h.fecha_hora) AS dia_semana_numero,
    HOUR(h.fecha_hora) + 1 AS periodo,

    g.generacion_real_kwh,
    d.demanda_total_kwh,
    d.demanda_regulada_kwh,
    d.demanda_no_regulada_kwh,
    disp.disponibilidad_real_kwh,

    p.precio_bolsa_nacional_cop_kwh,
    p.precio_bolsa_internacional_cop_kwh,
    p.precio_bolsa_tie_cop_kwh,

    g.generacion_real_kwh
        - d.demanda_total_kwh
        AS balance_generacion_demanda_kwh,

    disp.disponibilidad_real_kwh
        - g.generacion_real_kwh
        AS margen_disponibilidad_kwh,

    CASE
        WHEN disp.disponibilidad_real_kwh > 0
        THEN
            g.generacion_real_kwh
            / disp.disponibilidad_real_kwh
            * 100
    END AS utilizacion_disponibilidad_pct,

    CASE
        WHEN d.demanda_total_kwh > 0
        THEN
            g.generacion_real_kwh
            / d.demanda_total_kwh
            * 100
    END AS relacion_generacion_demanda_pct,

    CASE
        WHEN d.demanda_total_kwh IS NOT NULL
         AND p.precio_bolsa_nacional_cop_kwh IS NOT NULL
        THEN
            d.demanda_total_kwh
            * p.precio_bolsa_nacional_cop_kwh
    END AS valor_referencia_bolsa_cop

FROM horas h

LEFT JOIN generacion g
    ON h.fecha_hora = g.fecha_hora

LEFT JOIN demanda d
    ON h.fecha_hora = d.fecha_hora

LEFT JOIN disponibilidad disp
    ON h.fecha_hora = disp.fecha_hora

LEFT JOIN precio p
    ON h.fecha_hora = p.fecha_hora;

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_resumen_diario_sistema AS

WITH fechas AS (
    SELECT DISTINCT CAST(fecha_hora AS DATE) AS fecha
    FROM observatorio_dev.gold.fact_generacion_real

    UNION

    SELECT DISTINCT CAST(fecha_hora AS DATE)
    FROM observatorio_dev.gold.fact_demanda_real

    UNION

    SELECT DISTINCT CAST(fecha_hora AS DATE)
    FROM observatorio_dev.gold.fact_disponibilidad_planta

    UNION

    SELECT DISTINCT CAST(fecha_hora AS DATE)
    FROM observatorio_dev.gold.fact_precio_bolsa
),

generacion AS (
    SELECT
        CAST(fecha_hora AS DATE) AS fecha,
        SUM(generacion_real_kwh) AS generacion_kwh
    FROM observatorio_dev.gold.fact_generacion_real
    GROUP BY CAST(fecha_hora AS DATE)
),

demanda AS (
    SELECT
        CAST(fecha_hora AS DATE) AS fecha,
        SUM(demanda_real_kwh) AS demanda_kwh,

        SUM(
            CASE
                WHEN UPPER(tipo_mercado) = 'REGULADO'
                THEN demanda_real_kwh
                ELSE 0
            END
        ) AS demanda_regulada_kwh,

        SUM(
            CASE
                WHEN UPPER(tipo_mercado) = 'NO REGULADO'
                THEN demanda_real_kwh
                ELSE 0
            END
        ) AS demanda_no_regulada_kwh

    FROM observatorio_dev.gold.fact_demanda_real
    GROUP BY CAST(fecha_hora AS DATE)
),

disponibilidad AS (
    SELECT
        CAST(fecha_hora AS DATE) AS fecha,
        SUM(disponibilidad_real_kwh) AS disponibilidad_kwh
    FROM observatorio_dev.gold.fact_disponibilidad_planta
    GROUP BY CAST(fecha_hora AS DATE)
),

precio AS (
    SELECT
        CAST(fecha_hora AS DATE) AS fecha,

        AVG(precio_bolsa_nacional_cop_kwh)
            AS precio_nacional_promedio_cop_kwh,

        MIN(precio_bolsa_nacional_cop_kwh)
            AS precio_nacional_minimo_cop_kwh,

        MAX(precio_bolsa_nacional_cop_kwh)
            AS precio_nacional_maximo_cop_kwh,

        AVG(precio_bolsa_internacional_cop_kwh)
            AS precio_internacional_promedio_cop_kwh,

        AVG(precio_bolsa_tie_cop_kwh)
            AS precio_tie_promedio_cop_kwh

    FROM observatorio_dev.gold.fact_precio_bolsa
    GROUP BY CAST(fecha_hora AS DATE)
)

SELECT
    CAST(DATE_FORMAT(f.fecha, 'yyyyMMdd') AS INT) AS fecha_key,
    f.fecha,

    ROUND(g.generacion_kwh / 1000000, 3)
        AS generacion_gwh,

    ROUND(d.demanda_kwh / 1000000, 3)
        AS demanda_gwh,

    ROUND(d.demanda_regulada_kwh / 1000000, 3)
        AS demanda_regulada_gwh,

    ROUND(d.demanda_no_regulada_kwh / 1000000, 3)
        AS demanda_no_regulada_gwh,

    ROUND(dp.disponibilidad_kwh / 1000000, 3)
        AS disponibilidad_gwh,

    ROUND(
        (g.generacion_kwh - d.demanda_kwh) / 1000000,
        3
    ) AS margen_generacion_demanda_gwh,

    ROUND(
        100.0 * g.generacion_kwh
        / NULLIF(dp.disponibilidad_kwh, 0),
        2
    ) AS utilizacion_disponibilidad_pct,

    ROUND(p.precio_nacional_promedio_cop_kwh, 2)
        AS precio_nacional_promedio_cop_kwh,

    ROUND(p.precio_nacional_minimo_cop_kwh, 2)
        AS precio_nacional_minimo_cop_kwh,

    ROUND(p.precio_nacional_maximo_cop_kwh, 2)
        AS precio_nacional_maximo_cop_kwh,

    ROUND(p.precio_internacional_promedio_cop_kwh, 2)
        AS precio_internacional_promedio_cop_kwh,

    ROUND(p.precio_tie_promedio_cop_kwh, 2)
        AS precio_tie_promedio_cop_kwh,

    CASE WHEN g.fecha IS NOT NULL THEN TRUE ELSE FALSE END
        AS tiene_generacion,

    CASE WHEN d.fecha IS NOT NULL THEN TRUE ELSE FALSE END
        AS tiene_demanda,

    CASE WHEN dp.fecha IS NOT NULL THEN TRUE ELSE FALSE END
        AS tiene_disponibilidad,

    CASE WHEN p.fecha IS NOT NULL THEN TRUE ELSE FALSE END
        AS tiene_precio,

    CASE
        WHEN f.fecha = DATE '2026-07-14'
         AND d.fecha IS NULL
        THEN 'Sin información reportada por SIMEM'

        WHEN d.fecha IS NULL
        THEN 'Demanda no disponible'

        ELSE 'Disponible'
    END AS estado_demanda

FROM fechas f

LEFT JOIN generacion g
    ON f.fecha = g.fecha

LEFT JOIN demanda d
    ON f.fecha = d.fecha

LEFT JOIN disponibilidad dp
    ON f.fecha = dp.fecha

LEFT JOIN precio p
    ON f.fecha = p.fecha;


In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_actualizacion_fuentes AS

WITH cobertura AS (
    SELECT
        'Generación real' AS fuente,
        MIN(CAST(fecha_hora AS DATE)) AS fecha_minima,
        MAX(CAST(fecha_hora AS DATE)) AS fecha_maxima,
        COUNT(DISTINCT CAST(fecha_hora AS DATE))
            AS dias_disponibles,
        COUNT(*) AS registros
    FROM observatorio_dev.gold.fact_generacion_real

    UNION ALL

    SELECT
        'Demanda real',
        MIN(CAST(fecha_hora AS DATE)),
        MAX(CAST(fecha_hora AS DATE)),
        COUNT(DISTINCT CAST(fecha_hora AS DATE)),
        COUNT(*)
    FROM observatorio_dev.gold.fact_demanda_real

    UNION ALL

    SELECT
        'Disponibilidad',
        MIN(CAST(fecha_hora AS DATE)),
        MAX(CAST(fecha_hora AS DATE)),
        COUNT(DISTINCT CAST(fecha_hora AS DATE)),
        COUNT(*)
    FROM observatorio_dev.gold.fact_disponibilidad_planta

    UNION ALL

    SELECT
        'Precio de bolsa',
        MIN(CAST(fecha_hora AS DATE)),
        MAX(CAST(fecha_hora AS DATE)),
        COUNT(DISTINCT CAST(fecha_hora AS DATE)),
        COUNT(*)
    FROM observatorio_dev.gold.fact_precio_bolsa
)

SELECT
    fuente,
    fecha_minima,
    fecha_maxima,
    dias_disponibles,
    registros,
    DATEDIFF(CURRENT_DATE(), fecha_maxima) AS dias_rezago,
    CURRENT_TIMESTAMP() AS fecha_consulta
FROM cobertura;


In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_demanda_diaria_mercado AS

WITH demanda_horaria AS (
    SELECT
        fecha_key,
        CAST(fecha_hora AS DATE) AS fecha,
        fecha_hora,
        tipo_mercado,
        SUM(demanda_real_kwh) AS demanda_horaria_kwh
    FROM observatorio_dev.gold.fact_demanda_real
    GROUP BY
        fecha_key,
        CAST(fecha_hora AS DATE),
        fecha_hora,
        tipo_mercado
),

agentes_diarios AS (
    SELECT
        fecha_key,
        tipo_mercado,
        COUNT(DISTINCT agente_key) AS agentes_con_demanda
    FROM observatorio_dev.gold.fact_demanda_real
    GROUP BY
        fecha_key,
        tipo_mercado
),

demanda_diaria AS (
    SELECT
        fecha_key,
        fecha,
        tipo_mercado,

        SUM(demanda_horaria_kwh)
            AS demanda_total_kwh,

        AVG(demanda_horaria_kwh) / 1000
            AS demanda_promedio_mw,

        MAX(demanda_horaria_kwh) / 1000
            AS demanda_pico_mw,

        MIN(demanda_horaria_kwh) / 1000
            AS demanda_minima_mw,

        COUNT(DISTINCT fecha_hora)
            AS horas_con_datos

    FROM demanda_horaria

    GROUP BY
        fecha_key,
        fecha,
        tipo_mercado
)

SELECT
    d.fecha_key,
    d.fecha,

    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,
    calendario.semana_anio,
    calendario.dia_semana_numero,
    calendario.dia_semana_nombre,
    calendario.es_fin_semana,

    d.tipo_mercado,

    d.demanda_total_kwh / 1000000
        AS demanda_total_gwh,

    d.demanda_promedio_mw,
    d.demanda_pico_mw,
    d.demanda_minima_mw,

    agentes.agentes_con_demanda,
    d.horas_con_datos,

    CASE
        WHEN SUM(d.demanda_total_kwh)
             OVER (PARTITION BY d.fecha_key) > 0
        THEN
            d.demanda_total_kwh
            /
            SUM(d.demanda_total_kwh)
                OVER (PARTITION BY d.fecha_key)
            * 100
    END AS participacion_mercado_pct

FROM demanda_diaria d

INNER JOIN agentes_diarios agentes
    ON d.fecha_key = agentes.fecha_key
   AND d.tipo_mercado = agentes.tipo_mercado

INNER JOIN observatorio_dev.gold.dim_fecha calendario
    ON d.fecha_key = calendario.fecha_key;

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_demanda_diaria_agente AS

WITH demanda_agente AS (
    SELECT
        demanda.fecha_key,
        CAST(demanda.fecha_hora AS DATE) AS fecha,
        demanda.agente_key,
        demanda.tipo_mercado,

        SUM(demanda.demanda_real_kwh)
            AS demanda_total_kwh,

        AVG(demanda.demanda_real_kwh) / 1000
            AS demanda_promedio_mw,

        MAX(demanda.demanda_real_kwh) / 1000
            AS demanda_pico_mw,

        MIN(demanda.demanda_real_kwh) / 1000
            AS demanda_minima_mw,

        COUNT(DISTINCT demanda.fecha_hora)
            AS horas_con_datos,

        SUM(
            CASE
                WHEN demanda.demanda_real_kwh = 0
                THEN 1
                ELSE 0
            END
        ) AS horas_demanda_cero

    FROM observatorio_dev.gold.fact_demanda_real demanda

    GROUP BY
        demanda.fecha_key,
        CAST(demanda.fecha_hora AS DATE),
        demanda.agente_key,
        demanda.tipo_mercado
)

SELECT
    d.fecha_key,
    d.fecha,

    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,
    calendario.semana_anio,
    calendario.dia_semana_numero,
    calendario.dia_semana_nombre,
    calendario.es_fin_semana,

    d.agente_key,
    agente.codigo_agente,
    agente.nombre_agente,
    agente.actividad_agente,

    d.tipo_mercado,

    d.demanda_total_kwh / 1000000
        AS demanda_total_gwh,

    d.demanda_promedio_mw,
    d.demanda_pico_mw,
    d.demanda_minima_mw,

    d.horas_con_datos,
    d.horas_demanda_cero,

    CASE
        WHEN SUM(d.demanda_total_kwh) OVER (
            PARTITION BY
                d.fecha_key,
                d.tipo_mercado
        ) > 0
        THEN
            d.demanda_total_kwh
            /
            SUM(d.demanda_total_kwh) OVER (
                PARTITION BY
                    d.fecha_key,
                    d.tipo_mercado
            )
            * 100
    END AS participacion_agente_pct

FROM demanda_agente d

INNER JOIN observatorio_dev.gold.dim_agente agente
    ON d.agente_key = agente.agente_key

INNER JOIN observatorio_dev.gold.dim_fecha calendario
    ON d.fecha_key = calendario.fecha_key;

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_operacion_diaria_planta AS

WITH generacion AS (
    SELECT
        fecha_key,
        planta_key,

        SUM(generacion_real_kwh)
            AS generacion_total_kwh,

        AVG(generacion_real_kwh) / 1000
            AS generacion_promedio_mw,

        MAX(generacion_real_kwh) / 1000
            AS generacion_pico_mw,

        COUNT(DISTINCT fecha_hora)
            AS horas_con_generacion

    FROM observatorio_dev.gold.fact_generacion_real

    GROUP BY
        fecha_key,
        planta_key
),

disponibilidad AS (
    SELECT
        fecha_key,
        planta_key,

        SUM(disponibilidad_real_kwh)
            AS disponibilidad_total_kwh,

        AVG(disponibilidad_real_kwh) / 1000
            AS disponibilidad_promedio_mw,

        MAX(disponibilidad_real_kwh) / 1000
            AS disponibilidad_pico_mw,

        COUNT(DISTINCT fecha_hora)
            AS horas_con_disponibilidad

    FROM observatorio_dev.gold.fact_disponibilidad_planta

    GROUP BY
        fecha_key,
        planta_key
),

operacion AS (
    SELECT
        COALESCE(
            generacion.fecha_key,
            disponibilidad.fecha_key
        ) AS fecha_key,

        COALESCE(
            generacion.planta_key,
            disponibilidad.planta_key
        ) AS planta_key,

        generacion.generacion_total_kwh,
        generacion.generacion_promedio_mw,
        generacion.generacion_pico_mw,
        generacion.horas_con_generacion,

        disponibilidad.disponibilidad_total_kwh,
        disponibilidad.disponibilidad_promedio_mw,
        disponibilidad.disponibilidad_pico_mw,
        disponibilidad.horas_con_disponibilidad

    FROM generacion

    FULL OUTER JOIN disponibilidad
        ON generacion.fecha_key = disponibilidad.fecha_key
       AND generacion.planta_key = disponibilidad.planta_key
)

SELECT
    operacion.fecha_key,
    calendario.fecha,

    calendario.anio,
    calendario.trimestre,
    calendario.mes_numero,
    calendario.mes_nombre,
    calendario.anio_mes,
    calendario.anio_mes_nombre,
    calendario.semana_anio,
    calendario.dia_semana_numero,
    calendario.dia_semana_nombre,
    calendario.es_fin_semana,

    operacion.planta_key,
    planta.codigo_planta,
    planta.nombre_planta,
    planta.codigo_sic_agente,

    /*
    SIMEM entrega la capacidad efectiva neta en kW.
    Se exponen ambas unidades para facilitar el análisis.
    */
    planta.cap_efectiva_neta
        AS capacidad_efectiva_neta_kw,

    planta.cap_efectiva_neta / 1000
        AS capacidad_efectiva_neta_mw,

    planta.codigo_area_operativa,
    planta.codigo_sub_area_operativa,
    planta.tipo_despacho_recurso,
    planta.tipo_clasificacion,
    planta.tipo_generacion,
    planta.es_registro_inferido,

    /*
    Energía diaria.
    */
    operacion.generacion_total_kwh / 1000000
        AS generacion_total_gwh,

    operacion.disponibilidad_total_kwh / 1000000
        AS disponibilidad_total_gwh,

    /*
    Potencia media y máxima horaria.
    Como cada medición representa una hora:
    kWh / 1.000 = MW promedio durante esa hora.
    */
    operacion.generacion_promedio_mw,
    operacion.generacion_pico_mw,

    operacion.disponibilidad_promedio_mw,
    operacion.disponibilidad_pico_mw,

    operacion.horas_con_generacion,
    operacion.horas_con_disponibilidad,

    /*
    Diferencia entre disponibilidad y generación.
    Solo se calcula cuando ambas mediciones existen.
    */
    CASE
        WHEN operacion.generacion_total_kwh IS NOT NULL
         AND operacion.disponibilidad_total_kwh IS NOT NULL
        THEN
            (
                operacion.disponibilidad_total_kwh
                - operacion.generacion_total_kwh
            ) / 1000000
    END AS margen_disponibilidad_gwh,

    /*
    Porcentaje de la disponibilidad que fue generado.
    Puede superar 100% debido a inconsistencias
    o diferencias entre publicaciones de SIMEM.
    */
    CASE
        WHEN operacion.disponibilidad_total_kwh > 0
        THEN
            operacion.generacion_total_kwh
            / operacion.disponibilidad_total_kwh
            * 100
    END AS utilizacion_disponibilidad_pct,

    /*
    Factor de capacidad aproximado:

    Generación kWh
    ------------------------------- × 100
    Capacidad kW × horas observadas
    */
    CASE
        WHEN planta.cap_efectiva_neta > 0
         AND operacion.horas_con_generacion > 0
        THEN
            operacion.generacion_total_kwh
            /
            (
                planta.cap_efectiva_neta
                * operacion.horas_con_generacion
            )
            * 100
    END AS factor_capacidad_aproximado_pct,

    /*
    Cobertura técnica de las fuentes.
    */
    CASE
        WHEN operacion.generacion_total_kwh IS NULL
            THEN 'SOLO DISPONIBILIDAD'

        WHEN operacion.disponibilidad_total_kwh IS NULL
            THEN 'SOLO GENERACION'

        WHEN operacion.horas_con_generacion = 24
         AND operacion.horas_con_disponibilidad = 24
            THEN 'COMPLETA'

        ELSE 'PARCIAL'
    END AS estado_cobertura,

    /*
    Indica si el registro puede utilizarse para comparar
    generación contra disponibilidad.
    */
    CASE
        WHEN operacion.generacion_total_kwh IS NULL
          OR operacion.disponibilidad_total_kwh IS NULL
            THEN FALSE

        WHEN operacion.horas_con_generacion <> 24
          OR operacion.horas_con_disponibilidad <> 24
            THEN FALSE

        WHEN operacion.disponibilidad_total_kwh <= 0
            THEN FALSE

        ELSE TRUE
    END AS es_comparable_disponibilidad,

    /*
    Clasificación analítica de consistencia.
    Los registros no se eliminan ni se modifican.
    */
    CASE
        WHEN operacion.generacion_total_kwh IS NULL
          OR operacion.disponibilidad_total_kwh IS NULL
            THEN 'NO COMPARABLE'

        WHEN operacion.horas_con_generacion <> 24
          OR operacion.horas_con_disponibilidad <> 24
            THEN 'COBERTURA PARCIAL'

        WHEN operacion.generacion_total_kwh < 0
          OR operacion.disponibilidad_total_kwh < 0
            THEN 'VALOR NEGATIVO'

        WHEN operacion.disponibilidad_total_kwh = 0
         AND operacion.generacion_total_kwh > 0
            THEN 'DISPONIBILIDAD CERO CON GENERACION'

        WHEN operacion.disponibilidad_total_kwh = 0
         AND operacion.generacion_total_kwh = 0
            THEN 'SIN OPERACION'

        WHEN operacion.generacion_total_kwh
             > operacion.disponibilidad_total_kwh * 1.20
            THEN 'ANOMALIA MAYOR A 20%'

        WHEN operacion.generacion_total_kwh
             > operacion.disponibilidad_total_kwh * 1.05
            THEN 'ANOMALIA ENTRE 5% Y 20%'

        WHEN operacion.generacion_total_kwh
             > operacion.disponibilidad_total_kwh
            THEN 'VARIACION HASTA 5%'

        ELSE 'CONSISTENTE'
    END AS estado_consistencia,

    /*
    Bandera directa para controles de calidad.
    */
    CASE
        WHEN operacion.generacion_total_kwh
             > operacion.disponibilidad_total_kwh
        THEN TRUE
        ELSE FALSE
    END AS generacion_supera_disponibilidad

FROM operacion

LEFT JOIN observatorio_dev.gold.dim_planta planta
    ON operacion.planta_key = planta.planta_key

LEFT JOIN observatorio_dev.gold.dim_fecha calendario
    ON operacion.fecha_key = calendario.fecha_key;

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_generacion_diaria_tipo AS

WITH base AS (
    SELECT
        fecha_key,
        fecha,
        anio,
        trimestre,
        mes_numero,
        mes_nombre,
        anio_mes,
        anio_mes_nombre,
        semana_anio,
        dia_semana_numero,
        dia_semana_nombre,
        es_fin_semana,

        COALESCE(
            NULLIF(TRIM(tipo_generacion), ''),
            'SIN CLASIFICAR'
        ) AS tipo_generacion,

        planta_key,
        generacion_total_gwh,
        disponibilidad_total_gwh,

        capacidad_efectiva_neta_kw,
        horas_con_generacion,

        es_comparable_disponibilidad,
        estado_consistencia

    FROM observatorio_dev.gold.vw_operacion_diaria_planta

    WHERE generacion_total_gwh IS NOT NULL
),

agregado AS (
    SELECT
        fecha_key,
        fecha,
        anio,
        trimestre,
        mes_numero,
        mes_nombre,
        anio_mes,
        anio_mes_nombre,
        semana_anio,
        dia_semana_numero,
        dia_semana_nombre,
        es_fin_semana,
        tipo_generacion,

        SUM(generacion_total_gwh)
            AS generacion_total_gwh,

        /*
        Solo se suma disponibilidad cuando la comparación
        tiene cobertura completa y disponibilidad positiva.
        */
        SUM(
            CASE
                WHEN es_comparable_disponibilidad
                THEN disponibilidad_total_gwh
            END
        ) AS disponibilidad_comparable_gwh,

        SUM(
            CASE
                WHEN es_comparable_disponibilidad
                THEN generacion_total_gwh
            END
        ) AS generacion_comparable_gwh,

        COUNT(
    DISTINCT CASE
        WHEN generacion_total_gwh > 0
        THEN planta_key
    END
) AS plantas_generadoras,

        /*
        Capacidad de las plantas con generación observada.
        */
       SUM(
    CASE
        WHEN generacion_total_gwh > 0
         AND capacidad_efectiva_neta_kw IS NOT NULL
        THEN capacidad_efectiva_neta_kw / 1000
    END
) AS capacidad_observada_mw,

        /*
        Capacidad kW × horas = energía máxima teórica kWh.
        Se divide entre 1.000.000 para convertir a GWh.
        */
        SUM(
            CASE
                WHEN capacidad_efectiva_neta_kw > 0
                 AND horas_con_generacion > 0
                THEN
                    capacidad_efectiva_neta_kw
                    * horas_con_generacion
                    / 1000000
            END
        ) AS energia_maxima_teorica_gwh,

        COUNT(
            DISTINCT CASE
                WHEN estado_consistencia IN (
                    'ANOMALIA ENTRE 5% Y 20%',
                    'ANOMALIA MAYOR A 20%',
                    'DISPONIBILIDAD CERO CON GENERACION'
                )
                THEN planta_key
            END
        ) AS plantas_con_alerta,

        SUM(
            CASE
                WHEN estado_consistencia IN (
                    'ANOMALIA ENTRE 5% Y 20%',
                    'ANOMALIA MAYOR A 20%',
                    'DISPONIBILIDAD CERO CON GENERACION'
                )
                THEN 1
                ELSE 0
            END
        ) AS registros_con_alerta

    FROM base

    GROUP BY
        fecha_key,
        fecha,
        anio,
        trimestre,
        mes_numero,
        mes_nombre,
        anio_mes,
        anio_mes_nombre,
        semana_anio,
        dia_semana_numero,
        dia_semana_nombre,
        es_fin_semana,
        tipo_generacion
)

SELECT
    fecha_key,
    fecha,
    anio,
    trimestre,
    mes_numero,
    mes_nombre,
    anio_mes,
    anio_mes_nombre,
    semana_anio,
    dia_semana_numero,
    dia_semana_nombre,
    es_fin_semana,

    tipo_generacion,

    generacion_total_gwh,
    generacion_comparable_gwh,
    disponibilidad_comparable_gwh,

    plantas_generadoras,
    capacidad_observada_mw,
    energia_maxima_teorica_gwh,

    plantas_con_alerta,
    registros_con_alerta,

    CASE
        WHEN SUM(generacion_total_gwh)
             OVER (PARTITION BY fecha_key) > 0
        THEN
            generacion_total_gwh
            /
            SUM(generacion_total_gwh)
                OVER (PARTITION BY fecha_key)
            * 100
    END AS participacion_generacion_pct,

    CASE
        WHEN energia_maxima_teorica_gwh > 0
        THEN
            generacion_total_gwh
            / energia_maxima_teorica_gwh
            * 100
    END AS factor_capacidad_tipo_pct,

    CASE
        WHEN disponibilidad_comparable_gwh > 0
        THEN
            generacion_comparable_gwh
            / disponibilidad_comparable_gwh
            * 100
    END AS utilizacion_disponibilidad_tipo_pct

FROM agregado;

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_energia_embalsada_diaria AS

WITH relaciones AS (
    SELECT
        puente.planta_key,

        COUNT(DISTINCT puente.embalse_key)
            AS cantidad_embalses,

        MAX(
            CASE
                WHEN puente.es_relacion_unica
                THEN puente.embalse_key
            END
        ) AS embalse_key_unico,

        CONCAT_WS(
            ', ',
            SORT_ARRAY(
                COLLECT_SET(puente.codigo_embalse)
            )
        ) AS codigos_embalses_relacionados,

        CONCAT_WS(
            ', ',
            SORT_ARRAY(
                COLLECT_SET(puente.nombre_reservorio_fuente)
            )
        ) AS nombres_embalses_relacionados

    FROM observatorio_dev.gold.bridge_planta_embalse puente

    GROUP BY
        puente.planta_key
),

base AS (
    SELECT
        hecho.fecha_key,
        hecho.fecha_medicion,
        hecho.planta_key,
        hecho.energia_embalsada_kwh,
        hecho.version_seleccionada,
        hecho.prioridad_version,

        planta.codigo_planta,
        planta.nombre_planta,
        planta.codigo_sic_agente,

        COALESCE(relacion.cantidad_embalses, 0)
            AS cantidad_embalses,

        relacion.embalse_key_unico,
        relacion.codigos_embalses_relacionados,
        relacion.nombres_embalses_relacionados,

        embalse.codigo_embalse,
        embalse.nombre_embalse,
        embalse.latitud,
        embalse.longitud,
        embalse.coordenada_valida,
        embalse.requiere_revision_manual,

        CASE
            WHEN relacion.cantidad_embalses IS NULL
                THEN 'SIN RELACION'

            WHEN relacion.cantidad_embalses = 1
                THEN 'RELACION UNICA'

            ELSE 'RELACION MULTIPLE'
        END AS estado_relacion

    FROM observatorio_dev.gold.fact_energia_embalsada_planta hecho

    LEFT JOIN observatorio_dev.gold.dim_planta planta
        ON hecho.planta_key = planta.planta_key

    LEFT JOIN relaciones relacion
        ON hecho.planta_key = relacion.planta_key

    LEFT JOIN observatorio_dev.gold.dim_embalse embalse
        ON relacion.embalse_key_unico = embalse.embalse_key
),

unidades AS (
    SELECT
        fecha_key,
        fecha_medicion,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN CONCAT(
                'EMBALSE|',
                CAST(embalse_key_unico AS STRING)
            )

            WHEN estado_relacion = 'RELACION MULTIPLE'
            THEN CONCAT(
                'PLANTA_MULTIPLE|',
                CAST(planta_key AS STRING)
            )

            ELSE CONCAT(
                'SIN_RELACION|',
                CAST(planta_key AS STRING)
            )
        END AS unidad_analisis_key,

        estado_relacion,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN embalse_key_unico
        END AS embalse_key,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN codigo_embalse
        END AS codigo_embalse,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN nombre_embalse

            WHEN estado_relacion = 'RELACION MULTIPLE'
            THEN CONCAT(
                'ASIGNACION MULTIPLE - ',
                COALESCE(nombre_planta, codigo_planta)
            )

            ELSE CONCAT(
                'SIN RELACION - ',
                COALESCE(nombre_planta, codigo_planta)
            )
        END AS nombre_unidad_analisis,

        CASE
            WHEN estado_relacion <> 'RELACION UNICA'
            THEN planta_key
        END AS planta_contexto_key,

        CASE
            WHEN estado_relacion <> 'RELACION UNICA'
            THEN codigo_planta
        END AS codigo_planta_contexto,

        CASE
            WHEN estado_relacion <> 'RELACION UNICA'
            THEN nombre_planta
        END AS nombre_planta_contexto,

        codigo_sic_agente,
        cantidad_embalses,
        codigos_embalses_relacionados,
        nombres_embalses_relacionados,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN latitud
        END AS latitud,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN longitud
        END AS longitud,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN coordenada_valida
            ELSE FALSE
        END AS coordenada_valida,

        CASE
            WHEN estado_relacion = 'RELACION UNICA'
            THEN requiere_revision_manual
            ELSE TRUE
        END AS requiere_revision,

        planta_key,
        energia_embalsada_kwh,
        version_seleccionada,
        prioridad_version

    FROM base
),

agregado AS (
    SELECT
        unidad.fecha_key,
        unidad.fecha_medicion,

        calendario.anio,
        calendario.trimestre,
        calendario.mes_numero,
        calendario.mes_nombre,
        calendario.anio_mes,
        calendario.anio_mes_nombre,
        calendario.semana_anio,
        calendario.dia_semana_numero,
        calendario.dia_semana_nombre,
        calendario.es_fin_semana,

        unidad.unidad_analisis_key,
        unidad.estado_relacion,

        unidad.embalse_key,
        unidad.codigo_embalse,
        unidad.nombre_unidad_analisis,

        unidad.planta_contexto_key,
        unidad.codigo_planta_contexto,
        unidad.nombre_planta_contexto,
        unidad.codigo_sic_agente,

        unidad.cantidad_embalses,
        unidad.codigos_embalses_relacionados,
        unidad.nombres_embalses_relacionados,

        unidad.latitud,
        unidad.longitud,
        unidad.coordenada_valida,
        unidad.requiere_revision,

        SUM(unidad.energia_embalsada_kwh)
            AS energia_embalsada_kwh,

        SUM(unidad.energia_embalsada_kwh) / 1000000
            AS energia_embalsada_gwh,

        COUNT(DISTINCT unidad.planta_key)
            AS plantas_con_medicion,

        MAX(unidad.version_seleccionada)
            AS version_seleccionada,

        MAX(unidad.prioridad_version)
            AS prioridad_version

    FROM unidades unidad

    LEFT JOIN observatorio_dev.gold.dim_fecha calendario
        ON unidad.fecha_key = calendario.fecha_key

    GROUP BY
        unidad.fecha_key,
        unidad.fecha_medicion,

        calendario.anio,
        calendario.trimestre,
        calendario.mes_numero,
        calendario.mes_nombre,
        calendario.anio_mes,
        calendario.anio_mes_nombre,
        calendario.semana_anio,
        calendario.dia_semana_numero,
        calendario.dia_semana_nombre,
        calendario.es_fin_semana,

        unidad.unidad_analisis_key,
        unidad.estado_relacion,

        unidad.embalse_key,
        unidad.codigo_embalse,
        unidad.nombre_unidad_analisis,

        unidad.planta_contexto_key,
        unidad.codigo_planta_contexto,
        unidad.nombre_planta_contexto,
        unidad.codigo_sic_agente,

        unidad.cantidad_embalses,
        unidad.codigos_embalses_relacionados,
        unidad.nombres_embalses_relacionados,

        unidad.latitud,
        unidad.longitud,
        unidad.coordenada_valida,
        unidad.requiere_revision
),

comparacion AS (
    SELECT
        agregado.*,

        LAG(energia_embalsada_gwh) OVER (
            PARTITION BY unidad_analisis_key
            ORDER BY fecha_medicion
        ) AS energia_dia_anterior_gwh

    FROM agregado
)

SELECT
    fecha_key,
    fecha_medicion AS fecha,

    anio,
    trimestre,
    mes_numero,
    mes_nombre,
    anio_mes,
    anio_mes_nombre,
    semana_anio,
    dia_semana_numero,
    dia_semana_nombre,
    es_fin_semana,

    unidad_analisis_key,
    estado_relacion,

    embalse_key,
    codigo_embalse,
    nombre_unidad_analisis,

    planta_contexto_key,
    codigo_planta_contexto,
    nombre_planta_contexto,
    codigo_sic_agente,

    cantidad_embalses,
    codigos_embalses_relacionados,
    nombres_embalses_relacionados,

    latitud,
    longitud,
    coordenada_valida,
    requiere_revision,

    energia_embalsada_kwh,
    energia_embalsada_gwh,
    energia_dia_anterior_gwh,

    energia_embalsada_gwh
        - energia_dia_anterior_gwh
        AS variacion_diaria_gwh,

    CASE
        WHEN energia_dia_anterior_gwh > 0
        THEN
            (
                energia_embalsada_gwh
                - energia_dia_anterior_gwh
            )
            / energia_dia_anterior_gwh
            * 100
    END AS variacion_diaria_pct,

    CASE
        WHEN SUM(energia_embalsada_gwh)
             OVER (PARTITION BY fecha_key) > 0
        THEN
            energia_embalsada_gwh
            /
            SUM(energia_embalsada_gwh)
                OVER (PARTITION BY fecha_key)
            * 100
    END AS participacion_energia_dia_pct,

    plantas_con_medicion,
    version_seleccionada,
    prioridad_version,

    CASE
        WHEN estado_relacion = 'RELACION UNICA'
        THEN TRUE
        ELSE FALSE
    END AS es_asignacion_directa,

    MAX(fecha_medicion) OVER ()
        AS fecha_maxima_disponible

FROM comparacion;

In [0]:
%sql
CREATE OR REPLACE VIEW
observatorio_dev.gold.vw_resumen_energia_embalsada_diaria AS

WITH resumen AS (
    SELECT
        fecha_key,
        fecha,

        MAX(anio) AS anio,
        MAX(mes_numero) AS mes_numero,
        MAX(mes_nombre) AS mes_nombre,
        MAX(anio_mes) AS anio_mes,

        SUM(energia_embalsada_gwh)
            AS energia_total_gwh,

        SUM(
            CASE
                WHEN estado_relacion = 'RELACION UNICA'
                THEN energia_embalsada_gwh
                ELSE 0
            END
        ) AS energia_asignacion_directa_gwh,

        SUM(
            CASE
                WHEN estado_relacion = 'RELACION MULTIPLE'
                THEN energia_embalsada_gwh
                ELSE 0
            END
        ) AS energia_relacion_multiple_gwh,

        SUM(
            CASE
                WHEN estado_relacion = 'SIN RELACION'
                THEN energia_embalsada_gwh
                ELSE 0
            END
        ) AS energia_sin_relacion_gwh,

        COUNT(*) AS unidades_con_medicion,

        COUNT(
            CASE
                WHEN estado_relacion = 'RELACION UNICA'
                THEN 1
            END
        ) AS unidades_asignacion_directa,

        COUNT(
            CASE
                WHEN requiere_revision
                THEN 1
            END
        ) AS unidades_requieren_revision

    FROM observatorio_dev.gold.vw_energia_embalsada_diaria

    GROUP BY
        fecha_key,
        fecha
),

comparacion AS (
    SELECT
        resumen.*,

        LAG(energia_total_gwh) OVER (
            ORDER BY fecha
        ) AS energia_dia_anterior_gwh

    FROM resumen
)

SELECT
    *,

    energia_total_gwh
        - energia_dia_anterior_gwh
        AS variacion_diaria_gwh,

    CASE
        WHEN energia_dia_anterior_gwh > 0
        THEN
            (
                energia_total_gwh
                - energia_dia_anterior_gwh
            )
            / energia_dia_anterior_gwh
            * 100
    END AS variacion_diaria_pct,

    CASE
        WHEN energia_total_gwh > 0
        THEN
            energia_asignacion_directa_gwh
            / energia_total_gwh
            * 100
    END AS cobertura_asignacion_directa_pct,

    CASE
        WHEN energia_total_gwh > 0
        THEN
            energia_relacion_multiple_gwh
            / energia_total_gwh
            * 100
    END AS participacion_relacion_multiple_pct,

    CASE
        WHEN energia_total_gwh > 0
        THEN
            energia_sin_relacion_gwh
            / energia_total_gwh
            * 100
    END AS participacion_sin_relacion_pct,

    MAX(fecha) OVER ()
        AS fecha_maxima_disponible

FROM comparacion;

In [0]:
%sql
CREATE OR REPLACE VIEW observatorio_dev.gold.vw_dim_agente_powerbi AS
SELECT
    agente_key,
    codigo_agente,
    nombre_agente,
    nombre_agente_normalizado,
    actividad_agente
FROM observatorio_dev.gold.dim_agente
WHERE es_actual = TRUE;